<a href="https://colab.research.google.com/github/harshitasdev8/Myeloid-Project-Oda-Lab/blob/main/PDAC_Working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install lifelines

In [ ]:


import pandas as pd
from lifelines import CoxPHFitter

# ---------------------------------------------------------------------------
# STEP 1: Load raw data
# ---------------------------------------------------------------------------
RAW_PATH = "/content/denseDataOnlyDownload_PDAC_USCSXenaData.tsv"
df = pd.read_csv(RAW_PATH, sep="\t")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Define the Google Drive folder path
drive_output_dir = "/content/drive/MyDrive/PDAC_Data"

# Create the directory if it doesn't exist
os.makedirs(drive_output_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Clean — keep only primary tumor samples (TCGA code "01")
#          TCGA sample ID suffixes: 01 = primary tumor, 11 = normal tissue,
#          06 = metastatic. We only want primary tumor for this analysis.
# ---------------------------------------------------------------------------
df["sample_type_code"] = df["sample"].str.split("-").str[-1]
df_tumor = df[df["sample_type_code"] == "01"].copy()
print(f"Samples after filtering to primary tumor only: {df_tumor.shape[0]}")

GENES = ["CD68", "CD86", "CD163", "CD8A", "CD40"]
df_tumor = df_tumor.dropna(subset=GENES)
print(f"Samples after dropping missing gene expression: {df_tumor.shape[0]}")

Samples after filtering to primary tumor only: 185
Samples after dropping missing gene expression: 178


In [ ]:

# ---------------------------------------------------------------------------
# STEP 3: Build survival time + event indicator
#          event = 1 if patient died, 0 if censored (still alive at last follow-up)
#          time  = days_to_death if they died, else days_to_last_followup
# ---------------------------------------------------------------------------
df_tumor["event"] = (df_tumor["vital_status"] == "DECEASED").astype(int)
df_tumor["time"] = df_tumor["days_to_death"].fillna(df_tumor["days_to_last_followup"])

assert df_tumor["time"].isna().sum() == 0, "Some patients have no survival time at all — check raw data"

print(f"\nFinal analysis cohort: {df_tumor.shape[0]} patients "
      f"({df_tumor['event'].sum()} deaths, {(df_tumor['event']==0).sum()} censored)")



Final analysis cohort: 178 patients (93 deaths, 85 censored)


In [ ]:
# ---------------------------------------------------------------------------
# STEP 4: Build the CD163:CD8A ratio
#          Xena gene expression values are log2-transformed, so subtracting
#          log values is equivalent to a ratio: log(CD163/CD8A) = log(CD163) - log(CD8A)
# ---------------------------------------------------------------------------
df_tumor["CD163_CD8A_ratio"] = df_tumor["CD163"] - df_tumor["CD8A"]


In [ ]:

# ---------------------------------------------------------------------------
# STEP 5a: Univariate Cox models — each variable tested alone
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("UNIVARIATE COX MODELS (each variable tested alone)")
print("=" * 60)

univariate_results = []
for var in GENES + ["CD163_CD8A_ratio"]:
    cph = CoxPHFitter()
    data = df_tumor[[var, "time", "event"]].dropna()
    cph.fit(data, duration_col="time", event_col="event")
    row = cph.summary.loc[var]
    univariate_results.append({
        "variable": var,
        "HR": row["exp(coef)"],
        "p": row["p"],
        "C-index": cph.concordance_index_
    })

univ_df = pd.DataFrame(univariate_results)
print(univ_df.to_string(index=False))

# ---------------------------------------------------------------------------
# STEP 5b: Multivariate Cox model — all 5 genes together
#          This tests whether each gene predicts survival INDEPENDENTLY
#          of the others (e.g., does CD163 still matter once CD8A is accounted for?)
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("MULTIVARIATE COX MODEL (all 5 genes together)")
print("=" * 60)

cph_multi = CoxPHFitter()
cph_multi.fit(df_tumor[GENES + ["time", "event"]], duration_col="time", event_col="event")
print(cph_multi.summary[["coef", "exp(coef)", "p"]].to_string())
print(f"\nConcordance index: {cph_multi.concordance_index_:.3f}")

# ---------------------------------------------------------------------------
# STEP 5c: Multivariate Cox model using the CD163:CD8A ratio instead of
#          CD163 and CD8A separately, plus CD68/CD86/CD40
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("MULTIVARIATE COX MODEL (using CD163:CD8A ratio instead of separate genes)")
print("=" * 60)

ratio_vars = ["CD68", "CD86", "CD163_CD8A_ratio", "CD40"]
cph_ratio = CoxPHFitter()
cph_ratio.fit(df_tumor[ratio_vars + ["time", "event"]], duration_col="time", event_col="event")
print(cph_ratio.summary[["coef", "exp(coef)", "p"]].to_string())
print(f"\nConcordance index: {cph_ratio.concordance_index_:.3f}")

# ---------------------------------------------------------------------------
# STEP 6: Save cleaned data + results for reuse
# ---------------------------------------------------------------------------
df_tumor.to_excel(os.path.join(drive_output_dir, "pdac_clean_with_survival.xlsx"), index=False)
univ_df.to_excel(os.path.join(drive_output_dir, "pdac_univariate_results.xlsx"), index=False)
print(f"\nSaved cleaned dataset and univariate results to {drive_output_dir}/")


UNIVARIATE COX MODELS (each variable tested alone)
        variable       HR        p  C-index
            CD68 1.342115 0.008959 0.502351
            CD86 1.119183 0.175572 0.496633
           CD163 1.100971 0.132803 0.521056
            CD8A 1.018135 0.790857 0.481830
            CD40 1.290705 0.018678 0.535859
CD163_CD8A_ratio 1.144637 0.095398 0.537463

MULTIVARIATE COX MODEL (all 5 genes together)
               coef  exp(coef)         p
covariate                               
CD68       0.343902   1.410440  0.051479
CD86      -0.361598   0.696562  0.106147
CD163      0.191044   1.210513  0.170408
CD8A      -0.122588   0.884628  0.237266
CD40       0.302885   1.353758  0.033423

Concordance index: 0.566

MULTIVARIATE COX MODEL (using CD163:CD8A ratio instead of separate genes)
                      coef  exp(coef)         p
covariate                                      
CD68              0.334616   1.397404  0.055268
CD86             -0.280769   0.755203  0.038504
CD163_CD8A_ra